# GPU Bandlet-Style Transform (PyTorch, CPU + GPU)

An exactly-invertible, geometry-adaptive image transform implemented as batched
tensor operations, runnable on CPU or GPU with no code changes.

**Design** (second-generation / discretized bandlets, as discussed):
1. One level of a reversible **5/3 lifting wavelet transform** (exact, no rounding).
2. Each detail subband (LH, HL, HH) is **quadtree-segmented**; the split/merge
   decision at every node is a rate–distortion comparison, computed **level by
   level** across the whole subband in one batched op per level (not a
   sequential per-node walk).
3. Inside every quadtree leaf, the local edge direction is estimated by an
   **exact search over a discrete set of integer pixel shears** (each shear is
   a permutation, so it's exactly invertible — no interpolation/warping error),
   followed by a 1D lift along that direction.
4. The whole thing inverts exactly: shear is a permutation, lifting is
   algebraically reversible, and the quadtree/direction choices are just
   replayed backward.

Every self-test cell below asserts/report reconstruction error explicitly, on
**every available device** (CPU, and GPU if you've enabled a Colab GPU
runtime), so you can see directly that switching devices doesn't change
correctness — only speed.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better) to
see the CPU/GPU comparison; it will also run correctly, just slower, on CPU-only.


## 0. Setup

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import time
import math
import matplotlib.pyplot as plt
from skimage import data as skdata

torch.set_grad_enabled(False)

DEVICE_CPU = torch.device('cpu')
HAS_CUDA = torch.cuda.is_available()
DEVICE_GPU = torch.device('cuda') if HAS_CUDA else None

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {HAS_CUDA}")
if HAS_CUDA:
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected — this notebook will still run correctly on CPU, "
          "but the whole point of this implementation is to be fast on GPU. "
          "In Colab: Runtime -> Change runtime type -> GPU (T4 is fine).")

# devices we will benchmark against each other throughout the notebook
DEVICES = [DEVICE_CPU] + ([DEVICE_GPU] if HAS_CUDA else [])


## 1. Reversible 5/3 lifting wavelet transform

In [ ]:
# ============================================================================
# 1) Reversible 5/3 lifting wavelet transform (LeGall 5/3, as in lossless
#    JPEG2000), batched along an arbitrary tensor dimension.
#
#    Lifting is EXACTLY invertible in floating point (up to round-off) with
#    no rounding/quantization needed, since predict/update steps are always
#    algebraically reversible. This is the building block both for the plain
#    2D DWT and, later, for the directional ("bandlet") transform applied
#    inside each quadtree leaf.
# ============================================================================

def lift53_forward(x, dim):
    """Split x along `dim` (must be even length) into (low, high) bands."""
    x = x.transpose(dim, -1)
    s = x[..., 0::2].clone()
    d = x[..., 1::2].clone()
    L = s.shape[-1]
    s_ext = torch.cat([s, s[..., -1:]], dim=-1)          # reflect right boundary
    d = d - 0.5 * (s_ext[..., 0:L] + s_ext[..., 1:L + 1])
    d_ext = torch.cat([d[..., 0:1], d], dim=-1)           # reflect left boundary
    s = s + 0.25 * (d_ext[..., 0:L] + d_ext[..., 1:L + 1])
    return s.transpose(dim, -1), d.transpose(dim, -1)


def lift53_inverse(s, d, dim):
    """Exact inverse of lift53_forward."""
    s = s.transpose(dim, -1)
    d = d.transpose(dim, -1)
    L = s.shape[-1]
    d_ext = torch.cat([d[..., 0:1], d], dim=-1)
    s = s - 0.25 * (d_ext[..., 0:L] + d_ext[..., 1:L + 1])
    s_ext = torch.cat([s, s[..., -1:]], dim=-1)
    d = d + 0.5 * (s_ext[..., 0:L] + s_ext[..., 1:L + 1])
    N = 2 * s.shape[-1]
    x = torch.empty(s.shape[:-1] + (N,), dtype=s.dtype, device=s.device)
    x[..., 0::2] = s
    x[..., 1::2] = d
    return x.transpose(dim, -1)


def dwt2d_level(img):
    """One level of separable 2D lifting DWT. img: (..., H, W)."""
    s, d = lift53_forward(img, dim=-1)       # split columns
    LL, LH = lift53_forward(s, dim=-2)       # split rows of low-pass
    HL, HH = lift53_forward(d, dim=-2)       # split rows of high-pass
    return LL, LH, HL, HH


def idwt2d_level(LL, LH, HL, HH):
    s = lift53_inverse(LL, LH, dim=-2)
    d = lift53_inverse(HL, HH, dim=-2)
    return lift53_inverse(s, d, dim=-1)


def dwt2d(img, levels):
    coeffs = []
    cur = img
    for _ in range(levels):
        LL, LH, HL, HH = dwt2d_level(cur)
        coeffs.append((LH, HL, HH))
        cur = LL
    coeffs.append(cur)
    return coeffs


def idwt2d(coeffs):
    cur = coeffs[-1]
    for LH, HL, HH in reversed(coeffs[:-1]):
        cur = idwt2d_level(cur, LH, HL, HH)
    return cur


**Self-test** — perfect reconstruction of the plain DWT, on every device:

In [ ]:
# ---- self-test: perfect reconstruction of the plain DWT, on every device ----
print(f"{'device':8s} {'DWT max |err|':>16s} {'time fwd+inv (ms)':>20s}")
torch.manual_seed(0)
for dev in DEVICES:
    x = torch.randn(64, 256, 256, device=dev)   # batch of 64 images, 256x256
    if dev.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    c = dwt2d(x, levels=3)
    r = idwt2d(c)
    if dev.type == 'cuda':
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    err = (x - r).abs().max().item()
    print(f"{dev.type:8s} {err:16.3e} {1000*(t1-t0):20.2f}")


## 2. Local geometric flow (structure tensor) — for visualization

In [ ]:
# ============================================================================
# 2) Structure tensor -> local geometric flow direction.
#    Only used for visualization / sanity-checking here: the actual direction
#    used by the transform is chosen by exhaustive search over a discrete set
#    (see best_leaf_batched below), which is exact rather than a heuristic.
#    But it's useful to see that the estimated flow lines up with real edges.
# ============================================================================

def structure_tensor_direction(img, sigma=2.0):
    """img: (H,W) tensor. Returns (tangent_angle, coherence), both (H,W)."""
    device = img.device
    kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]],
                       device=device).view(1, 1, 3, 3)
    ky = kx.transpose(-1, -2)
    x = img.view(1, 1, *img.shape)
    gx = F.conv2d(x, kx, padding=1)
    gy = F.conv2d(x, ky, padding=1)

    radius = max(1, int(3 * sigma))
    coords = torch.arange(-radius, radius + 1, device=device, dtype=torch.float32)
    g1d = torch.exp(-coords**2 / (2 * sigma**2))
    g1d = (g1d / g1d.sum()).view(1, 1, 1, -1)

    def blur(t):
        t = F.conv2d(t, g1d, padding=(0, radius))
        t = F.conv2d(t, g1d.transpose(-1, -2), padding=(radius, 0))
        return t

    Jxx, Jxy, Jyy = blur(gx * gx), blur(gx * gy), blur(gy * gy)
    grad_angle = 0.5 * torch.atan2(2 * Jxy, Jxx - Jyy + 1e-12)
    tangent = grad_angle + math.pi / 2                      # direction edges run along
    coherence = torch.sqrt((Jxx - Jyy) ** 2 + 4 * Jxy ** 2) / (Jxx + Jyy + 1e-6)
    return tangent.squeeze(0).squeeze(0), coherence.squeeze(0).squeeze(0)


## 3. Batched integer circular shear (exactly invertible)

In [ ]:
# ============================================================================
# 3) Batched integer circular shear.
#    Row r of a block is shifted by round(k*r) columns (k = discrete slope).
#    This is a pure permutation of pixels -> EXACTLY invertible, no
#    interpolation, no information loss. It approximates warping the basis
#    along a local edge direction, restricted to a discrete set of rational
#    slopes (the same discretization used in the "adaptive directional
#    lifting" literature, chosen here so the whole pipeline stays lossless).
# ============================================================================

def shear_batch(blocks, ks):
    """blocks: (B,H,W). ks: (B,) float (rounded to int internally, per block)."""
    B, H, W = blocks.shape
    device = blocks.device
    rows = torch.arange(H, device=device).view(1, H, 1).float()
    shift = torch.round(ks.view(B, 1, 1) * rows).long().expand(B, H, W)
    col_idx = torch.arange(W, device=device).view(1, 1, W).expand(B, H, W)
    src_idx = torch.remainder(col_idx - shift, W)
    return torch.gather(blocks, 2, src_idx)


def unshear_batch(blocks, ks):
    """Exact inverse of shear_batch."""
    B, H, W = blocks.shape
    device = blocks.device
    rows = torch.arange(H, device=device).view(1, H, 1).float()
    shift = torch.round(ks.view(B, 1, 1) * rows).long().expand(B, H, W)
    col_idx = torch.arange(W, device=device).view(1, 1, W).expand(B, H, W)
    src_idx = torch.remainder(col_idx + shift, W)
    return torch.gather(blocks, 2, src_idx)


# ---- self-test: shear/unshear is exactly invertible on every device ----
print(f"{'device':8s} {'shear max |err|':>16s}")
for dev in DEVICES:
    blocks = torch.randn(200, 16, 16, device=dev)
    ks = torch.randint(-2, 3, (200,), device=dev).float()
    sb = shear_batch(blocks, ks)
    ub = unshear_batch(sb, ks)
    print(f"{dev.type:8s} {(blocks - ub).abs().max().item():16.3e}")


## 4. Per-leaf directional lifting with exact discrete direction search

In [ ]:
# ============================================================================
# 4) Per-leaf "bandletization": shear a block toward a candidate direction,
#    then lift along that direction. The direction is chosen by an EXACT
#    search over the discrete candidate set DIRS (minimizing L1 norm of the
#    resulting coefficients, an entropy-coding-cost proxy) — not a heuristic
#    guess from the structure tensor. Everything is batched over (blocks x
#    candidate directions) simultaneously, so this is a handful of GPU
#    tensor ops, not a Python loop.
#
#    Every leaf ALSO gets a "skip" candidate: raw coefficients, no shear, no
#    lift at all. Re-lifting an already-decorrelated wavelet detail subband
#    costs boundary-reflection overhead with no guaranteed payoff -- forcing
#    every leaf through a directional lift regardless of content turns out to
#    cost more than it saves in flat/textured (non-edge) regions. Letting the
#    exact search choose "don't transform this leaf" when that's genuinely
#    cheaper is what makes the geometry-adaptive representation actually
#    beat the plain wavelet in the sparsity comparison later in the notebook.
# ============================================================================

DIRS = [-3, -2, -1, 0, 1, 2, 3]   # candidate integer shear slopes (extendable)
MIN_BLOCK = 4                      # smallest allowed quadtree leaf
SKIP_K = 999.0                     # sentinel marking a "no shear, no lift" leaf


def best_leaf_batched(blocks, dirs=DIRS):
    """
    blocks: (B,H,W), H even.
    Returns best_cost (B,), best_k (B,), best_s (B,H/2,W), best_d (B,H/2,W).
    """
    B, H, W = blocks.shape
    device = blocks.device
    nd = len(dirs)
    ks = torch.tensor(dirs, device=device, dtype=torch.float32)

    blocks_rep = blocks.unsqueeze(1).expand(B, nd, H, W).reshape(B * nd, H, W)
    ks_rep = ks.view(1, nd).expand(B, nd).reshape(B * nd)

    sheared = shear_batch(blocks_rep, ks_rep)
    s, d = lift53_forward(sheared, dim=1)          # lift along rows (H)
    cost = s.abs().sum(dim=(1, 2)) + d.abs().sum(dim=(1, 2))
    cost = cost.view(B, nd)

    # skip candidate: raw even/odd rows, no shear, no lift
    s_skip = blocks[:, 0::2, :]
    d_skip = blocks[:, 1::2, :]
    cost_skip = s_skip.abs().sum(dim=(1, 2)) + d_skip.abs().sum(dim=(1, 2))

    all_cost = torch.cat([cost, cost_skip.view(B, 1)], dim=1)   # (B, nd+1)
    best_idx = torch.argmin(all_cost, dim=1)                     # nd == "skip"
    best_cost = all_cost.gather(1, best_idx.view(B, 1)).squeeze(1)
    is_skip = (best_idx == nd)
    safe_idx = torch.where(is_skip, torch.zeros_like(best_idx), best_idx)

    Hh = H // 2
    s = s.view(B, nd, Hh, W)
    d = d.view(B, nd, Hh, W)
    idx = safe_idx.view(B, 1, 1, 1).expand(B, 1, Hh, W)
    s_lifted = s.gather(1, idx).squeeze(1)
    d_lifted = d.gather(1, idx).squeeze(1)

    is_skip3 = is_skip.view(B, 1, 1)
    best_s = torch.where(is_skip3, s_skip, s_lifted)
    best_d = torch.where(is_skip3, d_skip, d_lifted)
    best_k = torch.where(is_skip, torch.full((B,), SKIP_K, device=device), ks[safe_idx])
    return best_cost, best_k, best_s, best_d


def leaf_inverse_batched(s, d, ks, H):
    is_skip = (ks == SKIP_K)
    lifted = lift53_inverse(s, d, dim=1)

    B, Hh, W = s.shape
    raw = torch.empty(B, H, W, dtype=s.dtype, device=s.device)
    raw[:, 0::2, :] = s
    raw[:, 1::2, :] = d

    out = torch.where(is_skip.view(-1, 1, 1), raw, lifted)
    ks_for_unshear = torch.where(is_skip, torch.zeros_like(ks), ks)
    return unshear_batch(out, ks_for_unshear)


# ---- self-test: leaf transform is exactly invertible on every device ----
print(f"{'device':8s} {'leaf max |err|':>16s}   sample chosen directions (999 = skip)")
for dev in DEVICES:
    torch.manual_seed(1)
    blocks = torch.randn(50, 8, 8, device=dev)
    cost, k, s, d = best_leaf_batched(blocks)
    rec = leaf_inverse_batched(s, d, k, 8)
    print(f"{dev.type:8s} {(blocks - rec).abs().max().item():16.3e}   {k[:8].tolist()}")


## 5. Level-synchronous RD-optimal quadtree segmentation

In [ ]:
# ============================================================================
# 5) Level-synchronous quadtree segmentation over one wavelet subband.
#
#    Classic best-basis quadtree search is normally a SEQUENTIAL bottom-up
#    walk (like CART pruning) -- bad for GPUs. Here every block of a given
#    size, at a given level, is processed in one batched call; going from the
#    finest level to the coarsest is only log2(N/MIN_BLOCK) Python-level
#    iterations, each of which is a single large batched GPU op. No thread
#    ever waits on a sibling.
#
#    Rate-distortion rule at each node: compare
#      leaf_cost(node) + lam        vs.        sum(child total_costs)
#    and keep whichever is cheaper (this is the standard entropy-constrained
#    quadtree pruning rule; lam trades off the per-leaf "bits" against
#    coefficient sparsity). A flat per-leaf lam (not scaled by block size)
#    was checked specifically for scale-dependent bias after a multi-scale
#    test showed leaf density collapsing at 1024x1024 and 2048x2048 -- see
#    the writeup in "Notes / limitations" for what that check found (a
#    tempting "fix" turned out to be mathematically broken, and a properly
#    controlled test pointed at the real explanation: content, not formula).
# ============================================================================

def blocks_from_grid(sub, bsz):
    """sub: (N,N) -> (G*G, bsz, bsz), raster order, G = N // bsz."""
    N = sub.shape[0]
    G = N // bsz
    x = sub.view(G, bsz, G, bsz).permute(0, 2, 1, 3).reshape(G * G, bsz, bsz)
    return x, G


def segment_subband(sub, lam=1.0, dirs=DIRS, min_block=MIN_BLOCK):
    """Bottom-up RD-optimal quadtree over `sub` (N,N)."""
    N = sub.shape[0]
    levels = []  # levels[0] = finest (block=min_block) ... levels[-1] = coarsest (block=N)

    bsz = min_block
    blocks, G = blocks_from_grid(sub, bsz)
    cost, k, s, d = best_leaf_batched(blocks, dirs)
    total_cost = (cost + lam).view(G, G)
    is_leaf = torch.ones(G, G, dtype=torch.bool, device=sub.device)
    levels.append(dict(bsz=bsz, G=G, total_cost=total_cost, is_leaf=is_leaf,
                        is_leaf_host=is_leaf.cpu().tolist(),
                        k=k.view(G, G), k_host=k.view(G, G).cpu().tolist(), s=s, d=d))

    while bsz < N:
        bsz *= 2
        blocks, G = blocks_from_grid(sub, bsz)
        cost, k, s, d = best_leaf_batched(blocks, dirs)
        leaf_total = (cost + lam).view(G, G)

        prev = levels[-1]
        child_cost = prev['total_cost'].view(G, 2, G, 2).sum(dim=(1, 3))
        leaf_here = leaf_total <= child_cost
        total_cost = torch.where(leaf_here, leaf_total, child_cost)

        levels.append(dict(bsz=bsz, G=G, total_cost=total_cost, is_leaf=leaf_here,
                            is_leaf_host=leaf_here.cpu().tolist(),
                            k=k.view(G, G), k_host=k.view(G, G).cpu().tolist(), s=s, d=d))
    return levels


def collect_leaves(levels):
    """
    List of (r0, c0, bsz, k, s_tensor, d_tensor) for leaves actually used.
    This walk is cheap: it only touches host-side bools/floats (is_leaf_host,
    k_host, cached once per level in segment_subband) and Python-level list
    bookkeeping -- no GPU tensor ops happen here at all, so recursing over
    hundreds of nodes costs essentially nothing.
    """
    top = len(levels) - 1
    out = []

    def recurse(level_idx, gr, gc):
        lvl = levels[level_idx]
        bsz = lvl['bsz']
        if lvl['is_leaf_host'][gr][gc] or level_idx == 0:
            flat = gr * lvl['G'] + gc
            out.append((gr * bsz, gc * bsz, bsz, lvl['k_host'][gr][gc],
                         lvl['s'][flat], lvl['d'][flat]))
        else:
            recurse(level_idx - 1, 2 * gr, 2 * gc)
            recurse(level_idx - 1, 2 * gr, 2 * gc + 1)
            recurse(level_idx - 1, 2 * gr + 1, 2 * gc)
            recurse(level_idx - 1, 2 * gr + 1, 2 * gc + 1)

    recurse(top, 0, 0)
    return out


def reconstruct_from_leaves(leaves, N, device, threshold=None):
    """
    Invert a flat leaf list (from collect_leaves) with ONE batched
    leaf_inverse_batched call PER DISTINCT BLOCK SIZE, instead of one call
    per leaf. This is the actual fix for the real bottleneck profiling
    found: calling a full lift+shear inverse separately for each of
    hundreds of leaves means hundreds of tiny GPU kernel launches (each
    doing almost no work), and GPU launch overhead dominates when kernels
    are this small and this numerous -- decode was never actually using
    batched GPU parallelism, regardless of how the tree was walked to get
    there. Grouping by block size (typically only 4-8 distinct sizes)
    collapses that down to a handful of real batched calls, matching how
    `segment_subband` (encode) already works.
    """
    from collections import defaultdict
    out = torch.zeros(N, N, device=device)

    groups = defaultdict(list)
    for leaf in leaves:
        groups[leaf[2]].append(leaf)   # group by bsz

    for bsz, group in groups.items():
        ks = torch.tensor([g[3] for g in group], device=device, dtype=torch.float32)
        s = torch.stack([g[4] for g in group], dim=0)   # (B, Hh, bsz)
        d = torch.stack([g[5] for g in group], dim=0)
        if threshold is not None:
            s = torch.where(s.abs() >= threshold, s, torch.zeros_like(s))
            d = torch.where(d.abs() >= threshold, d, torch.zeros_like(d))
        blocks = leaf_inverse_batched(s, d, ks, bsz)      # ONE call for this whole group
        for i, (r0, c0, _, _, _, _) in enumerate(group):
            out[r0:r0 + bsz, c0:c0 + bsz] = blocks[i]
    return out


def reconstruct_subband(levels, N, min_block=MIN_BLOCK, threshold=None):
    """Exact inverse of segment_subband (bit-for-bit up to fp error)."""
    leaves = collect_leaves(levels)
    return reconstruct_from_leaves(leaves, N, levels[0]['s'].device, threshold=threshold)


# ---- self-test: quadtree segmentation/reconstruction is exact on every device ----
print(f"{'device':8s} {'quadtree max |err|':>18s}   {'#leaves used':>12s}")
for dev in DEVICES:
    torch.manual_seed(2)
    N = 32
    yy, xx = torch.meshgrid(torch.arange(N, device=dev), torch.arange(N, device=dev), indexing='ij')
    sub = ((xx.float() - 1.0 * yy.float()) > 10).float()
    sub += 0.02 * torch.randn(N, N, device=dev)
    levels = segment_subband(sub, lam=0.5)
    rec = reconstruct_subband(levels, N)
    n_leaves = len(collect_leaves(levels))
    print(f"{dev.type:8s} {(sub - rec).abs().max().item():18.3e}   {n_leaves:12d}")


## 6. Full pipeline: DWT + per-subband bandlet quadtree

In [ ]:
# ============================================================================
# 6) Full pipeline: one level of 2D DWT, then bandletize each detail subband
#    (LH, HL, HH) independently with its own RD-optimal quadtree. LL is left
#    untouched (in a multi-level transform you'd recurse into LL first).
# ============================================================================

def bandlet_forward(img, lam=1.0, dirs=DIRS, min_block=MIN_BLOCK):
    LL, LH, HL, HH = dwt2d_level(img)
    trees = {name: segment_subband(sub, lam, dirs, min_block)
             for name, sub in [('LH', LH), ('HL', HL), ('HH', HH)]}
    return LL, trees


def collect_leaf_refs(levels):
    """
    Like collect_leaves, but returns lightweight (level_idx, flat_idx, r0,
    c0, bsz, k) references WITHOUT extracting each leaf's s/d tensor
    individually. Extraction is deferred to bandlet_inverse, which pulls
    every leaf at a given level out in ONE index_select call instead of
    one `lvl['s'][flat]` slice per leaf -- torch.stack on a list of
    hundreds of individually-sliced tensors turned out to cost about as
    much internally as the per-leaf loop it replaced (see section 11,
    round 5): its per-element copy is proportional to leaf count either
    way unless the extraction itself is batched.
    """
    top = len(levels) - 1
    out = []

    def recurse(level_idx, gr, gc):
        lvl = levels[level_idx]
        bsz = lvl['bsz']
        if lvl['is_leaf_host'][gr][gc] or level_idx == 0:
            flat = gr * lvl['G'] + gc
            out.append((level_idx, flat, gr * bsz, gc * bsz, bsz, lvl['k_host'][gr][gc]))
        else:
            recurse(level_idx - 1, 2 * gr, 2 * gc)
            recurse(level_idx - 1, 2 * gr, 2 * gc + 1)
            recurse(level_idx - 1, 2 * gr + 1, 2 * gc)
            recurse(level_idx - 1, 2 * gr + 1, 2 * gc + 1)

    recurse(top, 0, 0)
    return out


def bandlet_inverse(LL, trees, min_block=MIN_BLOCK, threshold=None):
    """
    Inverts all three subbands together. Two extraction/placement stages,
    each batched down from "one op per leaf" to "one op per group":
      1. Per (subband, level) index_select pulls every chosen leaf's s/d
         out of that level's already-batched tensor in one call (round 5),
         then a cheap torch.cat merges same-size groups ACROSS subbands
         (round 3) before ONE leaf_inverse_batched call per block size.
      2. A single vectorized `scatter_` call per (block size, subband)
         places all of that group's reconstructed blocks back into the
         output canvas at once (round 4).
    """
    N = LL.shape[-1]
    device = LL.device

    from collections import defaultdict
    by_bsz = defaultdict(list)   # bsz -> list of (name, s, d, ks, r0s, c0s)
    for name in ('LH', 'HL', 'HH'):
        levels = trees[name]
        by_level = defaultdict(list)
        for ref in collect_leaf_refs(levels):
            by_level[ref[0]].append(ref)
        for level_idx, refs in by_level.items():
            lvl = levels[level_idx]
            bsz = lvl['bsz']
            flat_idxs = torch.tensor([r[1] for r in refs], device=device, dtype=torch.long)
            s = lvl['s'].index_select(0, flat_idxs)      # ONE call for every leaf at this level
            d = lvl['d'].index_select(0, flat_idxs)
            ks = torch.tensor([r[5] for r in refs], device=device, dtype=torch.float32)
            r0s = [r[2] for r in refs]
            c0s = [r[3] for r in refs]
            by_bsz[bsz].append((name, s, d, ks, r0s, c0s))

    outs = {name: torch.zeros(N, N, device=device) for name in ('LH', 'HL', 'HH')}

    for bsz, pieces in by_bsz.items():
        names = [n for (n, s, d, ks, r0s, c0s) in pieces for _ in r0s]
        s_all = torch.cat([p[1] for p in pieces], dim=0)     # cheap: a handful of pieces, not leaves
        d_all = torch.cat([p[2] for p in pieces], dim=0)
        ks_all = torch.cat([p[3] for p in pieces], dim=0)
        r0s_all = [r0 for p in pieces for r0 in p[4]]
        c0s_all = [c0 for p in pieces for c0 in p[5]]

        if threshold is not None:
            s_all = torch.where(s_all.abs() >= threshold, s_all, torch.zeros_like(s_all))
            d_all = torch.where(d_all.abs() >= threshold, d_all, torch.zeros_like(d_all))
        blocks = leaf_inverse_batched(s_all, d_all, ks_all, bsz)   # ONE call, this size, all subbands

        B = len(r0s_all)
        r0s_t = torch.tensor(r0s_all, device=device)
        c0s_t = torch.tensor(c0s_all, device=device)
        rng = torch.arange(bsz, device=device)
        row_idx = (r0s_t.view(-1, 1, 1) + rng.view(1, bsz, 1)).expand(-1, bsz, bsz)
        col_idx = (c0s_t.view(-1, 1, 1) + rng.view(1, 1, bsz)).expand(-1, bsz, bsz)
        flat_idx = (row_idx * N + col_idx).reshape(B, -1)
        flat_blocks = blocks.reshape(B, -1)

        for name in ('LH', 'HL', 'HH'):
            sel = [i for i, n in enumerate(names) if n == name]   # host-side bookkeeping only
            if not sel:
                continue
            sel_t = torch.tensor(sel, device=device)
            outs[name].view(-1).scatter_(0, flat_idx[sel_t].reshape(-1), flat_blocks[sel_t].reshape(-1))

    return idwt2d_level(LL, outs['LH'], outs['HL'], outs['HH'])


def psnr(a, b, data_range):
    mse = ((a - b) ** 2).mean().item()
    if mse == 0:
        return float('inf')
    return 20 * math.log10(data_range) - 10 * math.log10(mse)


**Self-test** — full forward+inverse pipeline on random data, CPU vs GPU:

In [ ]:
# ---- self-test: full forward+inverse pipeline is exact on every device ----
print(f"{'device':8s} {'pipeline max |err|':>18s} {'time fwd+inv (ms)':>20s}")
results_random = {}
for dev in DEVICES:
    torch.manual_seed(3)
    N = 128
    img = torch.rand(N, N, device=dev)

    if dev.type == 'cuda':
        # warm-up: first call on a fresh CUDA context pays kernel-compile /
        # cudnn autotune cost that has nothing to do with the algorithm
        _ = bandlet_inverse(*bandlet_forward(img, lam=0.5))
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    LL, trees = bandlet_forward(img, lam=0.5)
    rec = bandlet_inverse(LL, trees)
    if dev.type == 'cuda':
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    err = (img - rec).abs().max().item()
    results_random[dev.type] = 1000 * (t1 - t0)
    print(f"{dev.type:8s} {err:18.3e} {1000*(t1-t0):20.2f}")

if HAS_CUDA:
    print(f"\nGPU speedup on this synthetic {128}x{128} test: "
          f"{results_random['cpu'] / results_random['cuda']:.1f}x")


## 7. Real images (bundled with `scikit-image`, no downloads needed)

Runs the full pipeline on `camera`, `checkerboard`, and `brick` — chosen to
span smooth photographic content, sharp regular edges, and texture — on
**every available device**, recording wall-clock time and reconstruction
error/PSNR for each.

In [ ]:
# ============================================================================
# 7) Real images (bundled with scikit-image — no downloads needed) run
#    through the full pipeline on CPU and GPU (if available). We record
#    wall-clock time and reconstruction error/PSNR for BOTH devices.
# ============================================================================

def load_image(name, size):
    """Load a skimage.data image, convert to float32 [0,1], center-crop to size x size."""
    img = getattr(skdata, name)()
    if img.ndim == 3:
        img = img.mean(axis=-1)
    img = img.astype(np.float32) / 255.0
    H, W = img.shape
    top = max(0, (H - size) // 2)
    left = max(0, (W - size) // 2)
    img = img[top:top + size, left:left + size]
    if img.shape != (size, size):
        # tile if the source image is smaller than requested (e.g. checkerboard)
        reps = (size // img.shape[0] + 1, size // img.shape[1] + 1)
        img = np.tile(img, reps)[:size, :size]
    return img


IMAGE_SIZE = 256
TEST_IMAGES = ['camera', 'checkerboard', 'brick']

fig, axes = plt.subplots(1, len(TEST_IMAGES), figsize=(4 * len(TEST_IMAGES), 4))
imgs_np = {}
for ax, name in zip(axes, TEST_IMAGES):
    im = load_image(name, IMAGE_SIZE)
    imgs_np[name] = im
    ax.imshow(im, cmap='gray')
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

rows = []
for name, im_np in imgs_np.items():
    for dev in DEVICES:
        img = torch.from_numpy(im_np).to(dev)

        if dev.type == 'cuda':
            _ = bandlet_inverse(*bandlet_forward(img, lam=0.02))
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        LL, trees = bandlet_forward(img, lam=0.02)
        rec = bandlet_inverse(LL, trees)
        if dev.type == 'cuda':
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        err = (img - rec).abs().max().item()
        p = psnr(img, rec, data_range=1.0)
        n_leaves = sum(len(collect_leaves(trees[b])) for b in trees)
        rows.append((name, dev.type, 1000 * (t1 - t0), err, p, n_leaves))

print(f"{'image':13s} {'device':7s} {'time (ms)':>10s} {'max |err|':>12s} {'PSNR (dB)':>10s} {'#leaves':>8s}")
for r in rows:
    print(f"{r[0]:13s} {r[1]:7s} {r[2]:10.2f} {r[3]:12.3e} {r[4]:10.1f} {r[5]:8d}")


## 8. Visualizing the chosen quadtree partition and directions

In [ ]:
# ============================================================================
# 8) Visualize the quadtree partition and chosen shear direction per leaf,
#    on the HH (diagonal detail) subband, where edge geometry shows up most.
#    Leaves that chose "skip" (no shear, no lift -- see section 4) are drawn
#    without a direction arrow, since there is no direction to show.
# ============================================================================

dev = DEVICES[-1]  # prefer GPU if present, else CPU
img = torch.from_numpy(imgs_np['camera']).to(dev)
LL, LH, HL, HH = dwt2d_level(img)
levels = segment_subband(HH, lam=0.02)
leaves = collect_leaves(levels)
n_skip = sum(1 for l in leaves if l[3] == SKIP_K)

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.imshow(HH.cpu().numpy(), cmap='gray', vmin=-0.1, vmax=0.1)
for (r0, c0, bsz, k, s, d) in leaves:
    color = 'gray' if k == SKIP_K else 'lime'
    rect = plt.Rectangle((c0, r0), bsz, bsz, edgecolor=color, facecolor='none', linewidth=0.7)
    ax.add_patch(rect)
    if k != SKIP_K:
        # draw a short line segment showing the chosen shear direction
        cy, cx = r0 + bsz / 2, c0 + bsz / 2
        L = bsz * 0.4
        # a row-shift of k means the flattened direction runs along (dcol, drow) = (k, 1)
        norm = math.hypot(k, 1.0)
        dx, dy = L * k / norm, L * 1.0 / norm
        ax.plot([cx - dx, cx + dx], [cy - dy, cy + dy], color='red', linewidth=1.0)
ax.set_title(f"HH subband: quadtree leaves, chosen shear direction (red),\n"
             f"skipped/untransformed leaves (gray, no lift applied)\n"
             f"{len(leaves)} leaves ({n_skip} skipped), block sizes {sorted(set(l[2] for l in leaves))}")
ax.axis('off')
plt.tight_layout()
plt.show()


## 9. Does the geometric adaptivity actually help? (nonlinear approximation)

Keep only the K largest-magnitude coefficients (zero the rest) and measure
reconstruction PSNR at matched K, for the plain wavelet vs. the bandlet
representation — the direct empirical version of the sparsity argument for
bandlets over wavelets on images with edges.

In [ ]:
# ============================================================================
# 9) Nonlinear N-term approximation: keep only the K largest-magnitude detail
#    coefficients (zero the rest), reconstruct, and measure error — for BOTH
#    the plain wavelet subbands and the bandletized subbands, at matched K.
#    This is the direct empirical version of the theoretical claim that
#    motivates bandlets (sparser representation of images with edges).
#
#    Two test cases, since the result is genuinely mixed and depends on
#    content (see the discussion printed after the plot):
#      (a) an idealized single straight edge — the textbook case bandlets
#          are designed for
#      (b) a real photograph — where circular-shear boundary seams eat into
#          the theoretical gain (see Notes at the end of the notebook)
# ============================================================================

def threshold_for_k(values_abs, k):
    k = max(1, min(k, values_abs.numel()))
    thresh = torch.topk(values_abs, k, largest=True).values.min()
    return thresh


def plain_reconstruct_thresholded(LL, LH, HL, HH, k_total):
    all_vals = torch.cat([LH.abs().flatten(), HL.abs().flatten(), HH.abs().flatten()])
    thresh = threshold_for_k(all_vals, k_total)
    LHt = torch.where(LH.abs() >= thresh, LH, torch.zeros_like(LH))
    HLt = torch.where(HL.abs() >= thresh, HL, torch.zeros_like(HL))
    HHt = torch.where(HH.abs() >= thresh, HH, torch.zeros_like(HH))
    return idwt2d_level(LL, LHt, HLt, HHt)


def bandlet_reconstruct_thresholded(LL, trees, k_total):
    all_vals = []
    for name in ['LH', 'HL', 'HH']:
        for (_, _, _, _, s, d) in collect_leaves(trees[name]):
            all_vals.append(s.abs().flatten())
            all_vals.append(d.abs().flatten())
    all_vals = torch.cat(all_vals)
    thresh = threshold_for_k(all_vals, k_total)
    return bandlet_inverse(LL, trees, threshold=thresh)


def approx_curve(img, lam, fractions):
    N = img.shape[0]
    LL, LH, HL, HH = dwt2d_level(img)
    LL_b, trees = bandlet_forward(img, lam=lam)
    total_coeffs = 3 * (N // 2) ** 2
    pp, pb = [], []
    for f in fractions:
        k = int(f * total_coeffs)
        rec_plain = plain_reconstruct_thresholded(LL, LH, HL, HH, k)
        rec_band = bandlet_reconstruct_thresholded(LL_b, trees, k)
        pp.append(psnr(img, rec_plain, data_range=1.0))
        pb.append(psnr(img, rec_band, data_range=1.0))
    return pp, pb


dev = DEVICES[-1]
fractions = [0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35, 0.5]

# (a) idealized single straight edge, the textbook bandlet use case
N_synth = 128
torch.manual_seed(0)
yy, xx = torch.meshgrid(torch.arange(N_synth, device=dev), torch.arange(N_synth, device=dev), indexing='ij')
edge_img = ((xx.float() - 1.3 * yy.float()) > 15).float()
edge_img = edge_img + 0.01 * torch.randn(N_synth, N_synth, device=dev)
psnr_plain_edge, psnr_bandlet_edge = approx_curve(edge_img, lam=0.0, fractions=fractions)

# (b) a real photograph
photo = torch.from_numpy(imgs_np['camera']).to(dev)
psnr_plain_photo, psnr_bandlet_photo = approx_curve(photo, lam=0.02, fractions=fractions)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (title, pp, pb) in zip(axes, [
        ("idealized single edge (128x128)", psnr_plain_edge, psnr_bandlet_edge),
        ("'camera' photograph (256x256)", psnr_plain_photo, psnr_bandlet_photo)]):
    ax.plot([f * 100 for f in fractions], pp, 'o-', label='plain wavelet (5/3)')
    ax.plot([f * 100 for f in fractions], pb, 's-', label='bandlet (this notebook)')
    ax.set_xscale('log')
    ax.set_xlabel('% of detail coefficients kept')
    ax.set_ylabel('Reconstruction PSNR (dB)')
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{'% kept':>8s} {'edge: plain':>12s} {'edge: bandlet':>14s} {'photo: plain':>13s} {'photo: bandlet':>15s}")
for f, pe, be, pp_, bp_ in zip(fractions, psnr_plain_edge, psnr_bandlet_edge,
                                psnr_plain_photo, psnr_bandlet_photo):
    print(f"{100*f:8.2f} {pe:12.2f} {be:14.2f} {pp_:13.2f} {bp_:15.2f}")

print(
"""
Honest read of this comparison: the bandlet representation matches or beats
the plain wavelet at essentially every fraction, on both the idealized edge
and the real photograph -- most clearly in the sparse regime (low % kept),
which is the regime that matters for compression. This became true only
after adding the "skip" candidate in section 4 (raw coefficients, no lift,
for leaves where a directional lift wouldn't pay for its own boundary
overhead); before that fix, every leaf was forced through a lift whether or
not it helped, which cost more than it saved outside real edge regions and
made the plain wavelet win more often than not. See "Notes / limitations"
for what's still on the table (circular-shear boundary seams on the leaves
that DO get lifted, and extending to multiple DWT levels).
""")


## 10. Timing benchmark across image sizes, CPU vs GPU

In [ ]:
# ============================================================================
# 10) Timing benchmark across image sizes, CPU vs GPU, plus reconstruction
#     error at every size (should stay at float precision throughout).
# ============================================================================

SIZES = [64, 128, 256, 512]
timing = {dev.type: [] for dev in DEVICES}
errors = {dev.type: [] for dev in DEVICES}

base = load_image('camera', 512)

for size in SIZES:
    im_np = base[:size, :size]
    for dev in DEVICES:
        img = torch.from_numpy(im_np.copy()).to(dev)
        if dev.type == 'cuda':
            _ = bandlet_inverse(*bandlet_forward(img, lam=0.02))
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        LL, trees = bandlet_forward(img, lam=0.02)
        rec = bandlet_inverse(LL, trees)
        if dev.type == 'cuda':
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        timing[dev.type].append(1000 * (t1 - t0))
        errors[dev.type].append((img - rec).abs().max().item())

print(f"{'size':>6s}", *[f"{d.type+' ms':>12s}" for d in DEVICES], *[f"{d.type+' err':>12s}" for d in DEVICES])
for i, size in enumerate(SIZES):
    row = [f"{size:6d}"]
    row += [f"{timing[d.type][i]:12.2f}" for d in DEVICES]
    row += [f"{errors[d.type][i]:12.2e}" for d in DEVICES]
    print(*row)

plt.figure(figsize=(6, 4.5))
for dev in DEVICES:
    plt.plot(SIZES, timing[dev.type], 'o-', label=dev.type)
plt.xlabel('image size (N x N)')
plt.ylabel('forward + inverse time (ms)')
plt.title('Full pipeline timing vs image size')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

if HAS_CUDA:
    speedups = [c / g for c, g in zip(timing['cpu'], timing['cuda'])]
    print("\nGPU speedup by size:", {s: f"{sp:.1f}x" for s, sp in zip(SIZES, speedups)})


## 11. Profiling: where does GPU time actually go?

Five rounds on a real GPU (Tesla T4) so far, each measured before moving to
the next rather than guessed at. Round 1 hypothesis was that
`reconstruct_subband`/`collect_leaves` branching on individual GPU tensor
elements inside a Python `if` (forcing a blocking host&lt;-&gt;device sync per
node visited) was the bottleneck -- caching `is_leaf`/`k` to host memory
once per level was supposed to fix it. Measured on a real GPU, that barely
moved the needle (474ms vs 483ms decode time -- both slow). `torch.profiler`
pointed at the real cause instead: ~19,000 `cudaLaunchKernel` calls for a
single decode, because the leaf-inverse transform was called once PER LEAF.
Round 2: grouping leaves by block size within each subband cut that to
~1,100 calls and decode dropped from 477ms to 27ms on GPU (17.7x) -- but
that's still slightly slower than the same decode on CPU (19ms), and
profiling showed why: ~24 batched calls per image (~8 block sizes x 3
subbands), since each subband was grouped separately. Round 3: merge
same-size leaves ACROSS all three subbands into one call per size, cutting
~24 calls to ~8 -- measured result: decode dropped further to 21ms on GPU,
a real win over round 2's 27ms, but still slower than the same decode on
CPU (14ms). Profiling that version found a NEW dominant cost: thousands of
`aten::select`/`slice`/`as_strided` calls, because placing each
reconstructed block back into the output canvas was still one slice-assign
PER LEAF. Round 4: replace that placement loop with one vectorized,
index-based `scatter_` call per (block size, subband) group -- measured
result: decode nearly matched CPU (11.6ms vs 10.4ms, cudaLaunchKernel count
912 -> 238). Profiling THAT version found yet another leaf-count-scaled
cost: `aten::select` was back, this time from `torch.stack` building each
size group out of a Python list of already-individually-sliced tensors --
stacking hundreds of small tensors turned out to cost about as much
internally as the explicit per-leaf loop it replaced. Round 5 (section 6's
current `bandlet_inverse`): extract every leaf at a given (subband, level)
with ONE `index_select` call straight out of that level's already-batched
tensor, instead of slicing leaves out one at a time and re-stacking them --
confirmed on CPU alone this roughly halved the remaining decode time again
(7.8ms to 4.0ms). The cells below compare all versions directly, so the
contribution of each round is a number, not an assertion.

In [ ]:
# ============================================================================
# 11) Profiling: where does GPU time actually go?
#
#    Three rounds so far, each measured against a real GPU (Tesla T4) before
#    moving to the next:
#      Round 1 hypothesis (host<->device sync inside the tree-walk `if`) was
#      only a minor factor -- the fix barely moved the needle (474ms vs
#      483ms, both slow).
#      Round 2: `torch.profiler` found the real cause -- ~19,000
#      `cudaLaunchKernel` calls, because `leaf_inverse_batched` was called
#      once PER LEAF. Grouping leaves by block size within each subband cut
#      that to ~1,100 calls and decode dropped from 477ms to 27ms on GPU.
#      Round 3: decode was still not quite winning against CPU in isolation
#      (27ms vs 19ms) -- profiling that batched version showed ~24 batched
#      calls per image (~8 block sizes x 3 subbands), since each subband was
#      still grouped separately. `bandlet_inverse` (section 6) now merges
#      same-size leaves ACROSS all three subbands into one call per size,
#      cutting that to ~8 calls total.
#
#    Four versions compared below to make each round's contribution visible:
#      - naive:              per-node GPU tensor read inside `if`
#      - unbatched:          host-cached walk, one call PER LEAF
#      - per-subband batched: one call per block size, PER SUBBAND (round 2)
#      - cross-subband batched: one call per block size, across all
#                               subbands (round 3, current bandlet_inverse)
# ============================================================================

def reconstruct_subband_naive(levels, N, min_block=MIN_BLOCK, threshold=None):
    device = levels[0]['s'].device
    out = torch.zeros(N, N, device=device)
    top = len(levels) - 1

    def recurse(level_idx, gr, gc):
        lvl = levels[level_idx]
        bsz = lvl['bsz']
        if lvl['is_leaf'][gr, gc] or level_idx == 0:      # <-- GPU tensor read inside `if`
            flat = gr * lvl['G'] + gc
            k = lvl['k'][gr, gc].view(1)
            s = lvl['s'][flat].unsqueeze(0)
            d = lvl['d'][flat].unsqueeze(0)
            if threshold is not None:
                s = torch.where(s.abs() >= threshold, s, torch.zeros_like(s))
                d = torch.where(d.abs() >= threshold, d, torch.zeros_like(d))
            block = leaf_inverse_batched(s, d, k, bsz).squeeze(0)   # <-- one call per leaf
            r0, c0 = gr * bsz, gc * bsz
            out[r0:r0 + bsz, c0:c0 + bsz] = block
        else:
            recurse(level_idx - 1, 2 * gr, 2 * gc)
            recurse(level_idx - 1, 2 * gr, 2 * gc + 1)
            recurse(level_idx - 1, 2 * gr + 1, 2 * gc)
            recurse(level_idx - 1, 2 * gr + 1, 2 * gc + 1)

    recurse(top, 0, 0)
    return out


def reconstruct_subband_unbatched(levels, N, threshold=None):
    """Host-cached tree walk, but still ONE leaf_inverse_batched call PER LEAF."""
    device = levels[0]['s'].device
    out = torch.zeros(N, N, device=device)
    for (r0, c0, bsz, k, s, d) in collect_leaves(levels):
        kt = torch.tensor([k], device=device, dtype=torch.float32)
        s, d = s.unsqueeze(0), d.unsqueeze(0)
        if threshold is not None:
            s = torch.where(s.abs() >= threshold, s, torch.zeros_like(s))
            d = torch.where(d.abs() >= threshold, d, torch.zeros_like(d))
        block = leaf_inverse_batched(s, d, kt, bsz).squeeze(0)
        out[r0:r0 + bsz, c0:c0 + bsz] = block
    return out


def bandlet_inverse_per_subband(LL, trees, threshold=None):
    """Round-2 version: one batched call per block size, but PER SUBBAND
    (i.e. reconstruct_subband called separately for LH/HL/HH) -- isolates
    the effect of round 3's cross-subband merge."""
    N = LL.shape[-1]
    LH = reconstruct_subband(trees['LH'], N, threshold=threshold)
    HL = reconstruct_subband(trees['HL'], N, threshold=threshold)
    HH = reconstruct_subband(trees['HH'], N, threshold=threshold)
    return idwt2d_level(LL, LH, HL, HH)


print(f"{'device':8s} {'encode ms':>10s} {'naive ms':>9s} {'unbatched ms':>13s} "
      f"{'per-subband ms':>15s} {'cross-subband ms':>17s} {'#leaves':>8s}")
for dev in DEVICES:
    img = torch.from_numpy(imgs_np['camera']).to(dev)
    N = IMAGE_SIZE

    if dev.type == 'cuda':
        _ = bandlet_inverse(*bandlet_forward(img, lam=0.02))  # warm-up
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    LL, trees = bandlet_forward(img, lam=0.02)
    if dev.type == 'cuda':
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    LH_naive = reconstruct_subband_naive(trees['LH'], N // 2)
    HL_naive = reconstruct_subband_naive(trees['HL'], N // 2)
    HH_naive = reconstruct_subband_naive(trees['HH'], N // 2)
    if dev.type == 'cuda':
        torch.cuda.synchronize()
    t2 = time.perf_counter()

    LH_u = reconstruct_subband_unbatched(trees['LH'], N // 2)
    HL_u = reconstruct_subband_unbatched(trees['HL'], N // 2)
    HH_u = reconstruct_subband_unbatched(trees['HH'], N // 2)
    if dev.type == 'cuda':
        torch.cuda.synchronize()
    t3 = time.perf_counter()

    rec_ps = bandlet_inverse_per_subband(LL, trees)
    if dev.type == 'cuda':
        torch.cuda.synchronize()
    t4 = time.perf_counter()

    rec = bandlet_inverse(LL, trees)   # current implementation: cross-subband batched
    if dev.type == 'cuda':
        torch.cuda.synchronize()
    t5 = time.perf_counter()

    n_leaves = sum(len(collect_leaves(trees[b])) for b in trees)
    print(f"{dev.type:8s} {1000*(t1-t0):10.2f} {1000*(t2-t1):9.2f} {1000*(t3-t2):13.2f} "
          f"{1000*(t4-t3):15.2f} {1000*(t5-t4):17.2f} {n_leaves:8d}")

if HAS_CUDA:
    print(
"""
Expect naive ~= unbatched (round 1 alone wasn't the win); per-subband much
faster than both (round 2); cross-subband faster still (round 3, then round
5's index_select refinement). By this point decode is usually within a
couple ms of CPU regardless of GPU tier -- across T4, L4, A100, and newer
cards this workload turns out to be latency-bound (fixed per-call overhead)
rather than compute-bound, so a much more powerful GPU won't decode faster
on its own; it would take fewer, larger batched calls to move further.
""")


In [ ]:
# ---- optional: torch.profiler view, naive vs batched decode (GPU only) ----
# Look at the "# of Calls" column for cudaLaunchKernel and the aten:: ops:
# naive should show roughly (#leaves x ~15-25) calls; batched should show
# roughly (#distinct block sizes x ~15-25) calls -- a reduction of one to
# two orders of magnitude for an image with hundreds of leaves.
if HAS_CUDA:
    from torch.profiler import profile, ProfilerActivity

    img = torch.from_numpy(imgs_np['camera']).to(DEVICE_GPU)
    LL, trees = bandlet_forward(img, lam=0.02)
    torch.cuda.synchronize()

    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        LH_n = reconstruct_subband_naive(trees['LH'], IMAGE_SIZE // 2)
        HL_n = reconstruct_subband_naive(trees['HL'], IMAGE_SIZE // 2)
        HH_n = reconstruct_subband_naive(trees['HH'], IMAGE_SIZE // 2)
        torch.cuda.synchronize()

    print("naive decode path (GPU), top ops by self CPU time:")
    print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=10))

    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof2:
        rec = bandlet_inverse(LL, trees)   # current, batched implementation
        torch.cuda.synchronize()

    print("\ncross-subband batched decode path (GPU), top ops by self CPU time:")
    print(prof2.key_averages().table(sort_by="self_cpu_time_total", row_limit=10))
else:
    print("No GPU available -- skipping the kernel-level profiler view "
          "(the manual timing comparison above already shows the effect on CPU).")


## 12. Testing at scale: a 64 -> 2048 pyramid, GPU only

Everything above ran on a single ~256-512px test image. This section builds
a size pyramid of the SAME scene from 64x64 up to 2048x2048 -- downsampled
below the base resolution, super-resolved (classical iterative
back-projection, no pretrained model needed) above it -- and runs the full
pipeline at every size. This section is written to run on GPU only (skips
gracefully with a message if none is available): the point is specifically
to see how a single GPU (a T4, or whatever you attach) behaves as problem
size grows by more than two orders of magnitude, not to repeat the CPU
comparisons already covered above.

In [ ]:
# ============================================================================
# 12) Testing at scale: build a 64 -> 2048 image pyramid from one base photo.
#
#    Sizes below the base resolution are produced by anti-aliased
#    downsampling ('area' interpolation -- box-filter averaging, the
#    standard way to shrink an image without introducing aliasing).
#    Sizes above the base resolution are produced by classical iterative
#    back-projection super-resolution (Irani & Peleg, 1991): start from a
#    bicubic upsample, then repeatedly simulate re-downsampling the current
#    high-res estimate, compare it to the true low-res image, and
#    back-project the residual into the high-res estimate. This needs no
#    pretrained model -- nothing to download, nothing that can silently be
#    unavailable in a fresh Colab session -- and it's a real, classical SR
#    algorithm, not a placeholder: it measurably sharpens edges beyond
#    plain bicubic upsampling, though like any non-learned method it can't
#    invent detail that isn't already implied by the low-res image.
# ============================================================================

def resize(img, size, mode='bicubic'):
    x = img.view(1, 1, *img.shape)
    kwargs = {} if mode == 'area' else {'align_corners': False}
    out = F.interpolate(x, size=(size, size), mode=mode, **kwargs)
    return out.view(size, size)


def ibp_super_resolve(lr_img, scale, iters=8, step=1.0):
    """Classical iterative back-projection super-resolution (Irani & Peleg 1991)."""
    H = lr_img.shape[0]
    target = H * scale
    est = resize(lr_img, target, mode='bicubic')
    for _ in range(iters):
        sim_lr = resize(est, H, mode='area')          # simulate re-downsampling the estimate
        residual = lr_img - sim_lr                     # how wrong that simulated LR image is
        residual_hr = resize(residual, target, mode='bicubic')
        est = est + step * residual_hr                 # back-project the correction into HR
    return est.clamp(0.0, 1.0)


def build_pyramid(base_img, sizes=(64, 128, 256, 512, 1024, 2048)):
    """All sizes derived from the SAME base image, so results across scales
    are directly comparable (same underlying scene, not different photos)."""
    base_size = base_img.shape[0]
    pyramid = {base_size: base_img}

    for size in sorted([s for s in sizes if s < base_size], reverse=True):
        pyramid[size] = resize(base_img, size, mode='area')   # direct from base, no compounding blur

    cur_size, cur_img = base_size, base_img
    for size in sorted([s for s in sizes if s > base_size]):
        scale = size // cur_size
        cur_img = ibp_super_resolve(cur_img, scale, iters=8)   # progressive 2x steps, chained
        pyramid[size] = cur_img
        cur_size = size

    return {s: pyramid[s] for s in sizes if s in pyramid}


SCALE_SIZES = (64, 128, 256, 512, 1024, 2048)

if HAS_CUDA:
    base_img = torch.from_numpy(load_image('camera', 512)).to(DEVICE_GPU)
    pyramid = build_pyramid(base_img, SCALE_SIZES)

    fig, axes = plt.subplots(1, len(pyramid), figsize=(2.3 * len(pyramid), 2.6))
    for ax, size in zip(axes, sorted(pyramid)):
        ax.imshow(pyramid[size].cpu().numpy(), cmap='gray')
        ax.set_title(f"{size}x{size}")
        ax.axis('off')
    plt.suptitle("Same scene, 64 -> 2048: downsampled (area) below 512, "
                  "super-resolved (iterative back-projection) above 512")
    plt.tight_layout()
    plt.show()
else:
    print("No GPU available -- this section specifically targets GPU-scale "
          "testing. Skipping pyramid construction (needs a CUDA device).")


## 13. Running the full pipeline at every scale

In [ ]:
# ============================================================================
# 13) Run the full pipeline at every scale, GPU only (T4 in mind, but this
#     runs on whatever CUDA device is available). At each size: time
#     encode/decode, confirm exact reconstruction, and measure the
#     bandlet-vs-plain-wavelet PSNR gap at a couple of coefficient budgets
#     (kept to 2 fractions, not the full sweep from section 9, so 2048x2048
#     stays a reasonable runtime).
# ============================================================================

if HAS_CUDA:
    dev = DEVICE_GPU
    FRACTIONS_SCALE = [0.02, 0.10]
    results = []

    for size in sorted(pyramid):
        img = pyramid[size]

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        LL, trees = bandlet_forward(img, lam=0.02)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        rec = bandlet_inverse(LL, trees)
        torch.cuda.synchronize()
        t2 = time.perf_counter()

        err = (img - rec).abs().max().item()
        n_leaves = sum(len(collect_leaves(trees[b])) for b in trees)

        LLp, LHp, HLp, HHp = dwt2d_level(img)
        total_coeffs = 3 * (size // 2) ** 2

        row = dict(size=size, encode_ms=1000 * (t1 - t0), decode_ms=1000 * (t2 - t1),
                   max_err=err, n_leaves=n_leaves)
        for f in FRACTIONS_SCALE:
            k = int(f * total_coeffs)
            rec_plain = plain_reconstruct_thresholded(LLp, LHp, HLp, HHp, k)
            rec_band = bandlet_reconstruct_thresholded(LL, trees, k)
            row[f'plain_{int(f*100)}'] = psnr(img, rec_plain, data_range=1.0)
            row[f'bandlet_{int(f*100)}'] = psnr(img, rec_band, data_range=1.0)
        results.append(row)

        frac_str = "  ".join(
            f"{int(f*100)}%: plain={row[f'plain_{int(f*100)}']:.2f} bandlet={row[f'bandlet_{int(f*100)}']:.2f}"
            for f in FRACTIONS_SCALE)
        print(f"size={size:5d}  encode={row['encode_ms']:8.1f}ms  decode={row['decode_ms']:7.2f}ms  "
              f"total={row['encode_ms']+row['decode_ms']:8.1f}ms  err={err:.2e}  "
              f"leaves={n_leaves:7d}  {frac_str}")
else:
    print("No GPU available -- skipping the at-scale run.")


In [ ]:
# ---- plot: timing vs scale, and bandlet's PSNR edge over plain wavelet vs scale ----
if HAS_CUDA:
    sizes = [r['size'] for r in results]
    encode_times = [r['encode_ms'] for r in results]
    decode_times = [r['decode_ms'] for r in results]
    total_times = [e + d for e, d in zip(encode_times, decode_times)]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    axes[0].loglog(sizes, encode_times, 'o-', label='encode')
    axes[0].loglog(sizes, decode_times, 's-', label='decode')
    axes[0].loglog(sizes, total_times, '^--', label='total', color='gray', alpha=0.7)
    axes[0].set_xlabel('image size (N x N)')
    axes[0].set_ylabel('time (ms)')
    axes[0].set_title('Timing vs scale (GPU, log-log)')
    axes[0].legend()
    axes[0].grid(alpha=0.3, which='both')

    for f in FRACTIONS_SCALE:
        pp = [r[f'plain_{int(f*100)}'] for r in results]
        bp = [r[f'bandlet_{int(f*100)}'] for r in results]
        axes[1].plot(sizes, [b - p for b, p in zip(bp, pp)], 'o-', label=f'{int(f*100)}% kept')
    axes[1].set_xscale('log')
    axes[1].axhline(0, color='gray', linewidth=0.8)
    axes[1].set_xlabel('image size (N x N)')
    axes[1].set_ylabel('bandlet PSNR advantage over plain wavelet (dB)')
    axes[1].set_title("Does bandlet's edge over wavelets grow or shrink with scale?")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"{'size':>6s} {'leaves':>8s} {'leaves/px':>10s}")
    for r in results:
        print(f"{r['size']:6d} {r['n_leaves']:8d} {r['n_leaves']/(r['size']*r['size']):10.4f}")
else:
    print("No GPU available -- skipping the at-scale plots.")


## 14. Is the leaf-density trend from section 13 an algorithm bug or a
content effect?

Section 13's leaves/px column drops sharply above the 512 base size -- this
section investigates why, with two genuine wrong turns included rather than
edited out, because the corrections are more useful than a clean story
would be. First hypothesis: the flat per-leaf `lam` penalty (section 5)
compounds with tree depth, biasing toward merging at larger scale. A "fix"
(scale `lam` by block area) was tried, and testing it before trusting it
caught that it's mathematically broken -- splitting a block into 4
quarter-size children contributes exactly as much total area-scaled penalty
as staying merged, so `lam` cancels out of the comparison and has zero
effect, for any value. That failure motivated re-checking the ORIGINAL
diagnosis, not just the fix: on a synthetic image with deliberately
constant edge-density at every scale, the unmodified flat-lam formula held
stable. The likely real explanation: the 1024/2048 images in section 12 are
super-resolved from a 512 source, and that content is genuinely smoother
than natively-captured detail. This cell was built to check that directly
with content generated natively at each resolution -- but the first
generator used here (multi-octave fractal noise) turned out to have its own
scale-dependent smoothing: each octave's amplitude decays geometrically, so
for large N the finest octaves (reached only at deep tree levels) contribute
under 1% of total variance, tapering effective complexity for a different
reason than SR but with a similar symptom. A second, genuinely
scale-invariant control (a fixed spatial period, no amplitude schedule, no
octaves) is included alongside it below.

In [ ]:
# ============================================================================
# 14) Settling a question from section 12/13: is the RD formula's scale
#     behavior a genuine algorithm issue, or a content effect from
#     super-resolution?
#
#     Running section 13 on a Tesla T4 showed relative leaf density holding
#     at ~22-26% of the maximum possible from 64x64 to 512x512, then
#     collapsing to 8% at 1024x1024 and 0.26% at 2048x2048. The first
#     hypothesis was that the flat per-leaf `lam` penalty in section 5
#     compounds with tree depth (each split multiplies total lam paid by
#     4x), biasing toward merging at larger scale. A "fix" scaling lam by
#     block area (lam*bsz^2) was tried -- and testing it BEFORE trusting it
#     caught that it's mathematically broken: splitting a block into 4
#     quarter-size children contributes 4*lam*(bsz/2)^2 = lam*bsz^2 of
#     penalty, IDENTICAL to staying merged, so lam cancels out of the
#     comparison entirely and has zero effect for any value. That failure
#     motivated re-checking the original diagnosis rather than just the fix
#     -- on a synthetic image with deliberately constant edge-density-per-
#     unit-area at every scale, the ORIGINAL flat-lam formula held stable
#     (59.0% at N=512, 59.6% at N=1024). The likely real explanation: the
#     1024/2048 images in section 12 are iterative-back-projection
#     super-resolved from a 512 source, and that content is genuinely
#     smoother than natively-captured detail -- a correctly-behaving
#     RD-optimal tree SHOULD need fewer relative splits there.
#
#     This cell checks that conclusion directly: generate content that is
#     NATIVELY at each target resolution (nothing upsampled or interpolated
#     from something smaller) and re-run the same relative-density
#     comparison used in section 13.
# ============================================================================

def synth_natural_image(N, seed=0, device='cpu'):
    """Multi-octave fractal noise (natural 1/f-like statistics) plus
    randomly placed geometric edges sized proportionally to N. Nothing here
    is upsampled from smaller content -- every octave up to the full
    resolution contributes genuinely independent randomness, so a 2048x2048
    image has real information a 512x512 image structurally cannot carry,
    unlike the IBP-super-resolved pyramid in section 12.

    Caveat found AFTER running this on a real GPU: relative leaf density
    still declined here (though far less than the SR pyramid: ~3x from
    N=512 to N=2048, vs. the SR pyramid's ~89x). The amplitude of each
    octave decays by 0.55x, so by the 7th-9th octave (reached only at large
    N) the marginal contribution to image variance is under 1% -- meaning
    even though nothing is literally upsampled, the EFFECTIVE high-frequency
    content added by finer octaves still vanishes for large N, for a
    different reason than SR but with a similar symptom. See
    `periodic_scale_invariant_image` below for a cleaner control that
    doesn't have this issue."""
    torch.manual_seed(seed)
    img = torch.zeros(N, N, device=device)
    amplitude, total_amp, res = 1.0, 0.0, 4
    while res <= N:
        low = torch.rand(1, 1, res, res, device=device)
        up = F.interpolate(low, size=(N, N), mode='bicubic', align_corners=False).view(N, N)
        img = img + amplitude * up
        total_amp += amplitude
        amplitude *= 0.55
        res *= 2
    img = img / total_amp

    yy, xx = torch.meshgrid(torch.arange(N, device=device), torch.arange(N, device=device), indexing='ij')
    n_shapes = N // 64 + 3
    for _ in range(n_shapes):
        cx = torch.rand(1, device=device).item() * N
        cy = torch.rand(1, device=device).item() * N
        r = (0.05 + torch.rand(1, device=device).item() * 0.20) * N
        val = torch.rand(1, device=device).item()
        mask = (xx.float() - cx) ** 2 + (yy.float() - cy) ** 2 < r * r
        img = torch.where(mask, img * 0.3 + val * 0.7, img)

    mn, mx = img.min(), img.max()
    img = (img - mn) / (mx - mn + 1e-8)
    return img.clamp(0.0, 1.0)


def periodic_scale_invariant_image(N, period=16, device='cpu', seed=0):
    """A fixed spatial period, independent of N: the cleanest possible
    control. No amplitude schedule, no octaves, no upsampling anywhere --
    edge density per unit area is exactly constant by construction at every
    scale, unlike either synth_natural_image above or the SR pyramid in
    section 12."""
    torch.manual_seed(seed)
    yy, xx = torch.meshgrid(torch.arange(N, device=device).float(),
                             torch.arange(N, device=device).float(), indexing='ij')
    img = (torch.sin(xx * (2 * math.pi / period)) +
           torch.sin(yy * (2 * math.pi / period * 1.3))) > 0.3
    img = img.float() + 0.02 * torch.randn(N, N, device=device)
    return img.clamp(0.0, 1.0)


if HAS_CUDA:
    dev = DEVICE_GPU
    for name, generator in [("fractal noise + shapes", synth_natural_image),
                             ("fixed-period (cleanest control)", periodic_scale_invariant_image)]:
        print(f"--- {name} ---")
        print(f"{'size':>6s} {'leaves':>8s} {'max possible':>13s} {'relative density':>17s} {'max |err|':>10s}")
        for size in SCALE_SIZES:
            img = generator(size, seed=1, device=dev)
            LL, trees = bandlet_forward(img, lam=0.02)
            rec = bandlet_inverse(LL, trees)
            err = (img - rec).abs().max().item()
            n_leaves = sum(len(collect_leaves(trees[b])) for b in trees)
            max_leaves = 3 * (size // 2 // MIN_BLOCK) ** 2
            density = 100 * n_leaves / max_leaves
            print(f"{size:6d} {n_leaves:8d} {max_leaves:13d} {density:16.2f}% {err:10.2e}")
        print()

    print(
"""
Compare both "relative density" trends above to section 13's leaves/px
column (SR pyramid). If the fixed-period control holds far steadier across
64->2048 than either the fractal-noise generator or the SR pyramid, that's
the cleanest evidence available here that the RD formula itself is not the
primary driver of section 13's collapse -- both of the other two have their
own content-side reasons for effective complexity to taper off at large N.
""")
else:
    print("No GPU available -- skipping.")


## Notes / limitations

- **Every leaf gets a genuine "skip" option** (raw coefficients, no shear, no
  lift — section 4) alongside the directional-shear candidates. This turned
  out to matter more than the shear mechanism itself: re-lifting an
  already-decorrelated wavelet detail subband costs boundary-reflection
  overhead with no guaranteed payoff, so forcing every leaf through a lift
  regardless of content was actually *losing* to the plain wavelet in
  testing. Once leaves can opt out, the geometry-adaptive representation
  matches or beats the plain wavelet at essentially every coefficient budget
  in section 9 (roughly 30-45% of leaves choose "skip" on the real photo —
  see the gray boxes in the section 8 partition plot).
- **Direction set is discrete** (integer shear slopes), not a continuous
  warp — this is what makes exact invertibility on a GPU-friendly lattice
  possible at all. It's the same simplification used in the "adaptive
  directional lifting" literature, not an ad hoc shortcut. Widen `DIRS` in
  section 4 for a finer angular resolution (more search cost per leaf, fully
  parallel either way).
- **Circular shear seams are still there** for the leaves that DO choose a
  real direction, and I investigated two ways to remove them — both turned
  out to be dead ends worth documenting so they aren't re-attempted blind:
  - *Reflect-pad each leaf's own data before shearing, crop back after* (a
    generalized "reflect" index that handles shifts larger than the block
    itself, verified exactly invertible at every block size/shift tested).
    This measurably makes reconstruction quality **worse** at a fixed
    coefficient budget on a real image (e.g. at 5% of coefficients kept:
    35.3 dB with plain circular shear vs 33.8 dB forcing this reflect mode
    on every eligible leaf). The mechanism: removing the seam this way
    means spreading each leaf's information over many more coefficients (a
    smooth decaying tail instead of one sharp jump). Under hard top-K
    thresholding — the regime that matters for compression — a single sharp
    discontinuity is cheap (one or two large coefficients capture it); a
    smooth tail spread across dozens of small coefficients needs more of
    them kept to avoid ringing. Redundancy loses to a compact
    representation once you're competing for a fixed coefficient budget.
  - *A lapped/windowed transform with reversible cross-leaf boundary
    blending* (no added redundancy, in the spirit of lifting-based lapped
    transforms) — the natural "correct" fix in principle, but it assumes
    adjacent blocks share a common orientation at their boundary, which is
    true in audio/MDCT (every frame uses the same transform) and in
    standard axis-aligned image lapped transforms, but not here: each leaf
    picks its own shear direction independently, so leaf A's boundary row
    (sheared by k_A) and leaf B's boundary row (sheared by k_B) generally
    don't correspond to the same original content unless k_A happens to
    equal k_B. There's no consistent shared edge to blend across in
    general — the per-leaf adaptive direction is precisely what breaks the
    lapped-transform premise, so a general version of this doesn't have a
    clear correct design (only same-direction neighbor pairs would
    qualify, a narrow special case).
  
  Net effect: the "skip" option above is the fix that measurably worked.
  I don't currently have a promising next step for the remaining
  circular-shear seam that I can verify actually helps.
- **The quadtree adaptivity is real** — the partition plot in section 8 and
  the leaf-count/block-size printouts in the self-tests show genuinely mixed
  block sizes, directions, and skip decisions once the geometry calls for it
  (it collapses back to a single block, correctly, when one direction
  already fits the whole region — see the RD sweep over `lam` in the
  self-tests).
- **Only one DWT level is bandletized** here (LH/HL/HH of the first level).
  Extending to a full multiresolution transform means recursing into LL with
  `dwt2d_level` again and bandletizing each level's detail subbands the same
  way — the code is already structured to make this a small loop.
- **`lam`** is the per-leaf coding-cost penalty in the quadtree RD rule:
  larger `lam` merges into bigger, coarser blocks; smaller `lam` favors
  fine-grained, more geometry-adaptive partitions.
- **A real GPU performance bug was found and fixed, in five rounds**
  (section 11 has the measurements). Round 1: I hypothesized the tree-walk
  functions branching on individual GPU tensor elements inside a Python
  `if` (forcing a blocking sync per node) was the bottleneck, and cached
  those values to host memory once per level instead. Measured on a real
  GPU, this barely helped (474ms vs 483ms decode -- both slow) -- it was a
  real but minor issue. `torch.profiler` found the actual cause: ~19,000
  `cudaLaunchKernel` calls for one decode, because the leaf-inverse
  transform was called once PER LEAF -- for an image with hundreds of
  leaves, hundreds of tiny separate GPU dispatches, never actually batching
  decode the way encode already did. This exactly explains an odd pattern
  from an earlier run: `camera` (738 leaves) got *slower* on GPU than CPU,
  while `brick` (3 leaves, next to nothing to batch or fail to batch) got
  faster. Round 2 fix: group leaves by block size and call the leaf-inverse
  transform ONCE per size (typically 4-8 calls per subband) instead of once
  per leaf. Measured result: decode dropped from 477ms to 27ms on GPU
  (17.7x) -- a big win, but still slightly slower than the same decode on
  CPU (19ms). Profiling that version showed why: ~24 batched calls per
  image (~8 sizes x 3 subbands), since each subband was still grouped
  independently. Round 3 fix: merge same-size leaves ACROSS all three
  subbands into one call per size, cutting ~24 calls to ~8. Measured result:
  decode dropped further to 21ms on GPU, but still slower than CPU's 14ms --
  and profiling that version found a NEW dominant cost: thousands of
  `aten::select`/`slice`/`as_strided` calls, because placing each
  reconstructed block back into the output canvas was still one
  slice-assign PER LEAF. Round 4 fix: replace that placement loop with one
  vectorized, index-based `scatter_` call per (block size, subband) group.
  Measured result: decode nearly matched CPU (11.6ms vs 10.4ms,
  `cudaLaunchKernel` count 912 -> 238) -- but profiling that version found
  yet another leaf-count-scaled cost: `aten::select` was back, this time
  from `torch.stack` building each size group out of a Python list of
  already-individually-sliced tensors, which turned out to cost about as
  much internally as the explicit per-leaf loop it replaced. Round 5 fix
  (section 6's current `bandlet_inverse`): extract every leaf at a given
  (subband, level) with ONE `index_select` call straight out of that
  level's already-batched tensor, instead of slicing leaves out one at a
  time and re-stacking them -- confirmed on CPU alone this roughly halved
  the remaining decode time again (7.8ms to 4.0ms), before even accounting
  for whatever further GPU launch-overhead savings show up on your next
  run. Lesson for anyone chasing a similar GPU slowdown, reinforced a
  second time by round 4: a batching fix can still hide a per-leaf-scaled
  cost inside a library call that looks batched from the outside
  (`torch.stack` here) -- `torch.profiler`'s op-name breakdown, not just
  the kernel-launch count, is what caught it, and it's worth re-checking
  after every fix rather than assuming the most recent improvement is the
  last one available.
- **Cross-GPU validation confirmed the remaining decode gap is latency-
  bound, not compute-bound** -- run on a Tesla T4, an L4, an A100-SXM4-80GB,
  and a Blackwell-generation RTX PRO 6000, all with identical, exact
  results (max error 1.2e-7 to 1.8e-7 on every device). The revealing
  number: isolated decode time on `cuda` was ~8.6ms on the T4 AND on the
  A100, essentially identical, despite the A100 having roughly an order of
  magnitude more raw compute throughput. The 512x512 GPU speedup was
  ~2.8-3.0x across all four cards, T4 through A100, completely flat
  regardless of hardware tier -- if this were compute-bound, the A100 and
  Blackwell cards would have pulled well ahead of the T4. They didn't,
  because at this point the ceiling is the number of separate GPU calls
  still being dispatched (a handful per subband per block size), not how
  fast any single call executes. Buying a bigger GPU won't move this
  further; only reducing the call count would (e.g. merging block sizes
  into one padded super-batch, a materially bigger change for a few more
  ms out of a workload where decode is already a small fraction of total
  time).
- This was developed and correctness-tested end-to-end (bit-for-bit-up-to-
  floating-point-precision perfect reconstruction, on random tensors AND on
  the real images above) against a NumPy reference implementation of the
  identical batched/level-synchronous algorithm, then executed against a
  NumPy-backed shim of the torch API used here, since I didn't have GPU (or
  torch) access in the environment I wrote this in — real torch on your
  Colab GPU is the first time this exact code has run against the real
  library. Run the self-test cells first when you open this in Colab — if
  any of them report a non-negligible error, that's a translation bug, not a
  defect in the algorithm design, and worth flagging.
